In [ ]:
# Freight Rate Prediction
## Machine Learning Engineering Assessment

This notebook covers:
1. Data loading and inspection
2. Exploratory data analysis
3. Data preprocessing
4. Feature engineering
5. Chronological model validation
6. Model development
7. Final model training
8. Validation and December predictions
9. Output verification

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

print("Libraries imported successfully.")

In [ ]:
DATA_DIR = "Data"

train = pd.read_csv(os.path.join(DATA_DIR, "train-test.csv"))
validation = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
template = pd.read_csv(
    os.path.join(DATA_DIR, "validation-predictions-template.csv")
)
december = pd.read_csv(
    os.path.join(DATA_DIR, "december-chart-inputs.csv")
)

print("TRAIN:", train.shape)
print("VALIDATION:", validation.shape)
print("TEMPLATE:", template.shape)
print("DECEMBER:", december.shape)

display(train.head())

In [ ]:
print("TRAINING DATA INFORMATION")
train.info()

print("\nVALIDATION DATA INFORMATION")
validation.info()

print("\nTRAINING MISSING VALUES")
display(train.isnull().sum())

print("\nVALIDATION MISSING VALUES")
display(validation.isnull().sum())

print("\nDUPLICATE TRAINING ROWS:", train.duplicated().sum())

print("\nTARGET VARIABLE SUMMARY")
display(train["posted_rate"].describe())

print("\nDATE COVERAGE")
print("Training:", train["date"].min(), "to", train["date"].max())
print("Validation:", validation["date"].min(), "to", validation["date"].max())

In [ ]:
# Convert date to datetime
train["date"] = pd.to_datetime(train["date"], errors="coerce")
validation["date"] = pd.to_datetime(validation["date"], errors="coerce")
december["date"] = pd.to_datetime(december["date"], errors="coerce")

# Target distribution
plt.figure(figsize=(9, 5))
plt.hist(train["posted_rate"], bins=50)
plt.title("Distribution of Posted Freight Rates")
plt.xlabel("Posted Rate ($)")
plt.ylabel("Number of Shipments")
plt.tight_layout()
plt.show()

# Distance versus posted rate
plt.figure(figsize=(9, 5))
plt.scatter(
    train["distance"],
    train["posted_rate"],
    alpha=0.25,
    s=10
)
plt.title("Freight Rate versus Distance")
plt.xlabel("Distance")
plt.ylabel("Posted Rate ($)")
plt.tight_layout()
plt.show()

# Average rate by equipment
equipment_rates = train.groupby("equipment")["posted_rate"].mean()

plt.figure(figsize=(8, 5))
equipment_rates.plot(kind="bar")
plt.title("Average Freight Rate by Equipment")
plt.xlabel("Equipment")
plt.ylabel("Average Posted Rate ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Monthly average rate
monthly_rates = (
    train.groupby(train["date"].dt.month)["posted_rate"]
    .mean()
)

plt.figure(figsize=(9, 5))
monthly_rates.plot(marker="o")
plt.title("Average Freight Rate by Month")
plt.xlabel("Month")
plt.ylabel("Average Posted Rate ($)")
plt.xticks(range(1, 11))
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("CORRELATION WITH POSTED RATE")

numeric_data = train.select_dtypes(include=np.number)

correlations = numeric_data.corr()["posted_rate"].sort_values(
    ascending=False
)

display(correlations)

print("\nEQUIPMENT COUNTS")
display(train["equipment"].value_counts())

print("\nAVERAGE RATE BY EQUIPMENT")
display(
    train.groupby("equipment")["posted_rate"]
    .agg(["count", "mean", "median", "std"])
)

print("\nMONTHLY RATE SUMMARY")
display(
    train.groupby(train["date"].dt.month)["posted_rate"]
    .agg(["count", "mean", "median"])
)

In [ ]:
# Remove identifiers, target, and questionable availability features
experiment_drop = [
    "load_id",
    "date",
    "posted_rate",
    "market_index",
    "quote_signal"
]

experiment_features = [
    col for col in train_features.columns
    if col not in experiment_drop
]

# Split by date
split_date = pd.Timestamp("2025-10-01")

experiment_train = train_features[
    train_features["date"] < split_date
].copy()

experiment_test = train_features[
    train_features["date"] >= split_date
].copy()

X_train_exp = experiment_train[experiment_features]
X_test_exp = experiment_test[experiment_features]

y_train_exp = experiment_train["posted_rate"]
y_test_exp = experiment_test["posted_rate"]

print("Training rows:", len(X_train_exp))
print("Testing rows:", len(X_test_exp))

print("Training period:",
      experiment_train["date"].min(),
      "to",
      experiment_train["date"].max())

print("Testing period:",
      experiment_test["date"].min(),
      "to",
      experiment_test["date"].max())

In [ ]:
# Remove identifiers, target, and questionable availability features
experiment_drop = [
    "load_id",
    "date",
    "posted_rate",
    "market_index",
    "quote_signal"
]

experiment_features = [
    col for col in train_features.columns
    if col not in experiment_drop
]

# Split by date
split_date = pd.Timestamp("2025-10-01")

experiment_train = train_features[
    train_features["date"] < split_date
].copy()

experiment_test = train_features[
    train_features["date"] >= split_date
].copy()

X_train_exp = experiment_train[experiment_features]
X_test_exp = experiment_test[experiment_features]

y_train_exp = experiment_train["posted_rate"]
y_test_exp = experiment_test["posted_rate"]

print("Training rows:", len(X_train_exp))
print("Testing rows:", len(X_test_exp))

print("Training period:",
      experiment_train["date"].min(),
      "to",
      experiment_train["date"].max())

print("Testing period:",
      experiment_test["date"].min(),
      "to",
      experiment_test["date"].max())

In [ ]:
numeric_features = X_train_exp.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train_exp.select_dtypes(
    exclude=np.number
).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

experiment_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.07,
        max_leaf_nodes=31,
        l2_regularization=2,
        random_state=RANDOM_STATE
    ))
])

print("Experiment model pipeline created.")

In [ ]:
# Logarithmic target transformation
y_train_log = np.log1p(y_train_exp)

experiment_model.fit(X_train_exp, y_train_log)

# Predict and convert back to original rate scale
experiment_predictions = np.expm1(
    experiment_model.predict(X_test_exp)
)

experiment_predictions = np.maximum(
    experiment_predictions, 0.01
)

mae = mean_absolute_error(
    y_test_exp, experiment_predictions
)

rmse = np.sqrt(
    mean_squared_error(y_test_exp, experiment_predictions)
)

r2 = r2_score(
    y_test_exp, experiment_predictions
)

print("OCTOBER INTERNAL VALIDATION RESULTS")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

In [ ]:
errors = np.abs(
    y_test_exp.values - experiment_predictions
)

print("ABSOLUTE ERROR DISTRIBUTION")

display(
    pd.Series(errors).describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

plt.figure(figsize=(9, 5))
plt.hist(errors, bins=50)
plt.title("October Validation Absolute Error Distribution")
plt.xlabel("Absolute Error ($)")
plt.ylabel("Number of Shipments")
plt.tight_layout()
plt.show()

# Actual versus predicted
plt.figure(figsize=(7, 7))
plt.scatter(
    y_test_exp,
    experiment_predictions,
    alpha=0.25,
    s=10
)

limit = max(
    y_test_exp.max(),
    experiment_predictions.max()
)

plt.plot([0, limit], [0, limit], linestyle="--")
plt.title("Actual versus Predicted Freight Rates")
plt.xlabel("Actual Rate ($)")
plt.ylabel("Predicted Rate ($)")
plt.tight_layout()
plt.show()

In [ ]:
final_features = [
    "pickup",
    "delivery",
    "distance",
    "equipment",
    "weight",
    "month",
    "day",
    "day_of_week"
]

X_final = train_features[final_features]
y_final = train["posted_rate"]

X_validation = validation_features[final_features]
X_december = december_features[final_features]

print("Final features:")
print(final_features)

print("\nFinal training shape:", X_final.shape)
print("Validation shape:", X_validation.shape)
print("December shape:", X_december.shape)

In [ ]:
final_numeric_features = X_final.select_dtypes(
    include=np.number
).columns.tolist()

final_categorical_features = X_final.select_dtypes(
    exclude=np.number
).columns.tolist()

final_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

final_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", final_numeric_transformer, final_numeric_features),
        ("categorical", final_categorical_transformer,
         final_categorical_features)
    ],
    remainder="drop"
)

final_model = Pipeline(steps=[
    ("preprocessor", final_preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.07,
        max_leaf_nodes=31,
        l2_regularization=2,
        random_state=RANDOM_STATE
    ))
])

# Train on all 48,000 development records
final_model.fit(X_final, np.log1p(y_final))

print("Final model trained successfully.")
print("Training records:", len(X_final))

In [ ]:
december_predictions = np.expm1(
    final_model.predict(X_december)
)

december_predictions = np.maximum(
    december_predictions, 0.01
)

december_output = december.copy()

december_output["predicted_rate"] = december_predictions

print("December prediction shape:", december_output.shape)
display(december_output.head())

assert len(december_output) == 31
assert december_output["predicted_rate"].notna().all()
assert (december_output["predicted_rate"] > 0).all()

december_output.to_csv(
    "december_predictions.csv",
    index=False
)

print("Saved december_predictions.csv")

In [ ]:
validation_check = pd.read_csv("validation_predictions.csv")
december_check = pd.read_csv("december_predictions.csv")

print("VALIDATION FILE")
display(validation_check.head())
print("Shape:", validation_check.shape)
print("Columns:", validation_check.columns.tolist())

print("\nDECEMBER FILE")
display(december_check.head())
print("Shape:", december_check.shape)
print("Columns:", december_check.columns.tolist())

print("\nVALIDATION CHECKS")
print("Correct row count:", len(validation_check) == 12000)
print("Unique load IDs:", validation_check["load_id"].is_unique)
print("Missing predictions:",
      validation_check["predicted_rate"].isna().sum())
print("Positive predictions:",
      (validation_check["predicted_rate"] > 0).all())

print("\nDECEMBER CHECKS")
print("Correct row count:", len(december_check) == 31)
print("Missing predictions:",
      december_check["predicted_rate"].isna().sum())
print("Positive predictions:",
      (december_check["predicted_rate"] > 0).all())

In [ ]:
!python score.py --predictions validation_predictions.csv --december-predictions december_predictions.csv

In [ ]:
from IPython.display import Image, display

chart_path = "scorer_results/candidate_december.png"

if os.path.exists(chart_path):
    display(Image(filename=chart_path))
else:
    print("December chart has not been generated yet.")